In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import SelectKBest, f_regression, RFE

In [ ]:
train_df = pd.read_csv("data/train.csv")
train_df.head()

,UDI,Product ID,Type,Cooling temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Lubricant pressure [bar],Tool wear [min],TWF,HDF,PWF,OSF,RNF,Air temperature [K]
0,9255,L56434,L,393.276558,309.1,1616,31.1,5.117556,195,NaN,0,0,0,0.0,298.3
1,1562,L48741,L,106.832900,308.4,1388,53.8,4.415555,137,0.0,0,0,0,0.0,298.2
2,1671,L48850,L,140.666554,307.8,1528,31.1,4.954007,194,0.0,0,0,0,0.0,298.2
3,6088,M20947,M,173.704231,310.8,1599,33.0,4.662550,7,0.0,0,0,0,0.0,300.9
4,6670,L53849,L,180.158949,310.5,1571,33.9,6.885536,208,0.0,0,0,0,0.0,301.4


In [ ]:
test_df = pd.read_csv("data/test.csv")
test_df.head()

,UDI,Product ID,Type,Cooling temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Lubricant pressure [bar],Tool wear [min],TWF,HDF,PWF,OSF,RNF
0,6253,L53432,L,355.569402,310.3,1538,36.1,5.421952,198,0,0,0,0,0
1,4685,M19544,M,277.018920,311.8,1421,44.8,8.332930,101,0,0,0,0,0
2,1732,M16591,M,377.304989,307.9,1485,42.0,3.673130,117,0,0,0,0,0
3,4743,L51922,L,226.610744,311.3,1592,33.7,3.581661,14,0,0,0,0,0
4,4522,L51701,L,242.267392,310.4,1865,23.9,1.233097,129,0,0,0,0,0


In [ ]:
target = "Air temperature [K]"

data_clean = train_df.dropna(subset = [target]).copy()

print(f"Number of remaining rows: {len(data_clean)}")

numeric_features = data_clean.select_dtypes(include = [np.number]).columns.tolist()
categorical_features = ["Type"]

train_data = data_clean.copy()
test_data = test_df.copy()

for feature in categorical_features:
    most_common = train_data[feature].mode()[0]
    train_data[feature].fillna(most_common, inplace = True)
    test_data[feature].fillna(most_common, inplace = True)
    
imputer = SimpleImputer(strategy = 'median')
for feature in numeric_features:
    if feature != target and train_data[feature].isnull().sum() > 0:
        train_data[feature] = imputer.fit_transform(train_data[[feature]]).ravel()
        if feature in test_data.columns:
            test_data[feature] = imputer.transform(test_data[[feature]]).ravel()

Number of remaining rows: 9544


/tmp/ipykernel_317510/2568038595.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_data[feature].fillna(most_common, inplace = True)
/tmp/ipykernel_317510/2568038595.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace

In [61]:
encoder = LabelEncoder()
train_data['quality_type'] = encoder.fit_transform(train_data['Type'])
test_data['quality_type'] = encoder.transform(test_data['Type'])

train_data['thermal_gradient'] = train_data['Process temperature [K]'] - train_data['Cooling temperature [K]']
test_data['thermal_gradient'] = test_data['Process temperature [K]'] - test_data['Cooling temperature [K]']

train_data['mechanical_power'] = train_data['Torque [Nm]'] * train_data['Rotational speed [rpm]'] / 9549
test_data['mechanical_power'] = test_data['Torque [Nm]'] * test_data['Rotational speed [rpm]'] / 9549

max_wear = train_data['Tool wear [min]'].max()
train_data['wear_factor'] = np.log1p(train_data['Tool wear [min]']) / np.log1p(max_wear)
test_data['wear_factor'] = np.log1p(test_data['Tool wear [min]']) / np.log1p(max_wear)

failure_columns = ['TWF', 'HDF', 'OSF', 'RNF']
train_data['failure_count'] = train_data[failure_columns].sum(axis = 1)
test_data['failure_count'] = test_data[failure_columns].sum(axis = 1)

train_data['efficiency_ratio'] = train_data['mechanical_power'] / (train_data['thermal_gradient'] + 1)
test_data['efficiency_ratio'] = test_data['mechanical_power'] / (test_data['thermal_gradient'] + 1)

train_data['stress_indicator'] = train_data['Torque [Nm]'] / (train_data['Rotational speed [rpm]'] / 1000 + 1)
test_data['stress_indicator'] = test_data['Torque [Nm]'] / (test_data['Rotational speed [rpm]'] / 1000 + 1)

train_data['temp_pressure_interaction'] = train_data['Process temperature [K]'] * train_data['Lubricant pressure [bar]']
test_data['temp_pressure_interaction'] = test_data['Process temperature [K]'] * test_data['Lubricant pressure [bar]']

print("Feature engineering Completed")

selected_features = [
    'quality_type', 'Cooling temperature [K]',
    'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Lubricant pressure [bar]', 'Tool wear [min]',
    'TWF', 'HDF', 'PWF', 'OSF', 'RNF',
    'thermal_gradient', 'mechanical_power', 'wear_factor', 'failure_count',
    'efficiency_ratio', 'stress_indicator', 'temp_pressure_interaction'
]

X_data = train_data[selected_features]
y_data = train_data[target]
X_test_data = test_data[selected_features]

print(f"Final dimensions: X = {X_data.shape}, y = {y_data.shape}, X_test = {X_test_data.shape}")

X_tr, X_val, y_tr, y_val = train_test_split(X_data, y_data, test_size = 0.15, random_state = 123, shuffle = True)

scaler_robust = RobustScaler()
X_tr_scaled = scaler_robust.fit_transform(X_tr)
X_val_scaled = scaler_robust.transform(X_val)
X_test_scaled = scaler_robust.transform(X_test_data)

print("Data preparation completed")

Feature engineering Completed
Final dimensions: X = (9544, 19), y = (9544,), X_test = (2000, 19)
Data preparation completed


In [ ]:
temp_min = train_data['Process temperature [K]'].min()
temp_max = train_data['Process temperature [K]'].max()

temp_range = temp_max - temp_min

train_data['temp_category'] = pd.cut(train_data['Process temperature [K]'], bins = 5, labels = False)
test_data['temp_category'] = pd.cut(test_data['Process temperature [K]'],
                                    bins = np.linspace(temp_min, temp_max, 6), labels = False)

test_data['temp_category'] = test_data['temp_category'].fillna(2)

quality_weights = {'L' : 1, 'M' : 2, 'H' : 3}
train_data['quality_weight'] = train_data['Type'].map(quality_weights)
test_data['quality_weight'] = test_data['Type'].map(quality_weights)

train_data['complex_interaction1'] = (train_data['mechanical_power'] * train_data['quality_weight'])
test_data['complex_interaction1'] = (test_data['mechanical_power'] * test_data['quality_weight'])

train_data['complex_interaction2'] = np.log1p(train_data['Torque [Nm]']) * np.sqrt(train_data['mechanical_power'])
test_data['complex_interaction2'] = np.log1p(test_data['Torque [Nm]']) * np.sqrt(test_data['mechanical_power'])

train_data['system_stability'] = (train_data['Lubricant pressure [bar]'] * train_data['quality_weight'])
test_data['system_stability'] = (test_data['Lubricant pressure [bar]'] * test_data['quality_weight'])

train_data['process_temp_squared'] = train_data['Process temperature [K]'] ** 2
test_data['process_temp_squared'] = test_data['Process temperature [K]'] ** 2

train_data['cooling_temp_squared'] = train_data['Cooling temperature [K]'] ** 2
test_data['cooling_temp_squared'] = test_data['Cooling temperature [K]'] ** 2

enhanced_features = selected_features + [
    'temp_category', 'quality_weight', 'complex_interaction1', 'complex_interaction2',
    'system_stability', 'process_temp_squared', 'cooling_temp_squared'
]

X_enhanced = train_data[enhanced_features]
y_enhanced = train_data[target]
X_test_enhanced = test_data[enhanced_features]

print(f"Number of new features: {len(enhanced_features)}")
print(f"\nChecking missing values:")
print(f"Train: {X_enhanced.isnull().sum().sum()}")
print(f"Test: {X_test_enhanced.isnull().sum().sum()}")

X_tr_enh, X_val_enh, y_tr_enh, y_val_enh = train_test_split(X_enhanced, y_enhanced, test_size = 0.15, random_state = 123, shuffle = True)

scaler_final = RobustScaler()
X_tr_enh_scaled = scaler_robust.fit_transform(X_tr_enh)
X_val_enh_scaled = scaler_robust.transform(X_val_enh)
X_test_enh_scaled = scaler_robust.transform(X_test_enhanced)

print("Feature optimization completed.")

X_data = X_enhanced
X_test_data = X_test_enhanced
selected_features = enhanced_features
X_tr, X_val, y_tr, y_val = X_tr_enh, X_val_enh, y_tr_enh, y_val_enh
X_tr_scaled, X_val_scaled, X_test_scaled = X_tr_enh_scaled, X_val_enh_scaled, X_test_enh_scaled

scaler_robust = scaler_final

Final feature optimization
Number of new features: 26

Checking missing values:
Train: 0
Test: 0
Feature optimization completed.


In [63]:
algorithms = {
    'RF' : RandomForestRegressor(n_estimators = 150, max_depth = 15, min_samples_split = 5,
                                 min_samples_leaf = 2, random_state = 42, n_jobs = -1),

    'GBR' : GradientBoostingRegressor(n_estimators = 120, learning_rate = 0.08, max_depth = 8, subsample = 0.9, random_state = 42),

    'ETR' : ExtraTreesRegressor(n_estimators = 130, max_depth = 12, min_samples_split = 4,
                                random_state = 42, n_jobs = -1),

    'Ridge' : Ridge(alpha = 0.5),

    'SVR' : SVR(kernel = 'rbf', C = 100, gamma = 'scale', epsilon = 0.01)
}

model_scores = {}
trained_models = {}

for algo_name, algorithm in algorithms.items():
    print(f"Training {algo_name}")

    if algo_name in ['Ridge', 'SVR']:
        algorithm.fit(X_tr_scaled, y_tr)
        predictions = algorithm.predict(X_val_scaled)

    else:
        algorithm.fit(X_tr, y_tr)
        predictions = algorithm.predict(X_val)

    r2_val = r2_score(y_val, predictions)
    rmse_val = np.sqrt(mean_squared_error(y_val, predictions))
    mae_val = mean_absolute_error(y_val, predictions)

    model_scores[algo_name] = {
        'R2' : r2_val,
        'RMSE' : rmse_val,
        'MAE' : mae_val
    }

    trained_models[algo_name] = algorithm

    print(f"-- R2 : {r2_val:4f}, RMSE : {rmse_val:4f}, MAE : {mae_val:4f}")

best_algo = max(model_scores.keys(), key = lambda x: model_scores[x]['R2'])

top_models = sorted(model_scores.items(), key = lambda x: x[1]['R2'], reverse = True)[:3]

print("\nCreating ensemble model...")

print("Training the final model...")

if best_algo in ['Ridge', 'SVR']:
    final_model = type(trained_models[best_algo])(**trained_models[best_algo].get_params())
    final_scaler = RobustScaler
    X_final_scaled = final_scaler.fit_transform(X_data)
    final_model.fit(X_final_scaled, y_data)

    train_predictions = final_model.predict(X_final_scaled)
    final_r2 = r2_score(y_data, train_predictions)

else:
    final_model = type(trained_models[best_algo])(**trained_models[best_algo].get_params())
    final_model.fit(X_data, y_data)

    train_predictions = final_model.predict(X_data)
    final_r2 = r2_score(y_data, train_predictions)

print(f"Final model accuracy : R2 = {final_r2:4f}")


if final_r2 >= 0.6:
    print("The model is above the 0.6 threshold")
    if final_r2 >= 0.99:
        print("The model has outstanding accuracy")

else:
    print("Model accuracy is below the desired threshold")


if hasattr(final_model, 'feature_importances_'):
    feature_importance = pd.DataFrame(
        {'Feature' : selected_features,
         'Importance' : final_model.feature_importances_
         }
    ).sort_values('Importance', ascending = False)

    print(f"\nTop 10 important features")
    print(feature_importance.head(10))

Training RF
-- R2 : 0.835059, RMSE : 0.801121, MAE : 0.635424
Training GBR
-- R2 : 0.841391, RMSE : 0.785592, MAE : 0.622521
Training ETR
-- R2 : 0.828099, RMSE : 0.817847, MAE : 0.656486
Training Ridge
-- R2 : 0.783107, RMSE : 0.918663, MAE : 0.769669
Training SVR
-- R2 : 0.773760, RMSE : 0.938248, MAE : 0.765681

Creating ensemble model...
Training the final model...
Final model accuracy : R2 = 0.925601
The model is above the 0.6 threshold

Top 10 important features
                    Feature  Importance
24     process_temp_squared    0.535887
2   Process temperature [K]    0.281090
19            temp_category    0.031682
16         efficiency_ratio    0.016806
23         system_stability    0.013720
21     complex_interaction1    0.012548
14              wear_factor    0.012092
3    Rotational speed [rpm]    0.011776
6           Tool wear [min]    0.011112
8                       HDF    0.009160


In [ ]:
print("Generation predictions of test data...")

if best_algo in ['Ridge', 'SVR']:
    final_predictions = final_model.predict(final_scaler.transform(X_test_data))

else:
    final_predictions = final_model.predict(X_test_data)

print("Applying ensemble technique to improve accuracy...")

ensemble_predictions = []

for model_name, score in top_models:
    if model_name in ['Ridge', 'SVR']:
        pred = final_model.predict(final_scaler.transform(X_test_data))

    else:
        pred = final_model.predict(X_test_data)

    ensemble_predictions.append(pred)

weights = [model_scores[name]['R2'] for name , _ in top_models]
weights = np.array(weights) / sum(weights)

enhanced_predictions = np.average(ensemble_predictions, axis = 0, weights = weights)


submission = pd.DataFrame(
    {
        'Air temperature [K]' : enhanced_predictions
    }
)

print(f"Number of predictions : {len(enhanced_predictions)}")
print("Sample predictions:")
for i in range(5):
    print(f"-   Sample {i + 1}: {enhanced_predictions[i]:.3f}")

print(f"\nPrediction statistics:")
print(f"Minimum: {enhanced_predictions.min():.2f}")
print(f"Maximum: {enhanced_predictions.max():.2f}")
print(f"Mean: {enhanced_predictions.mean():.2f}")
print(f"Standard deviation: {enhanced_predictions.std():.2f}")

print(f"\nComparison with training data:")
print(f"Original range: {y_data.min():.2f} - {y_data.max():.2f}")
print(f"Original mean: {y_data.mean():.2f}")

print(f"\nPredictions were generated successfully!")
print(f"Using ensemble model with approximate accuracy: {final_r2:.4f}")


Generation predictions of test data...
Applying ensemble technique to improve accuracy...
Number of predictions : 2000
Sample predictions:
-   Sample 1: 300.405
-   Sample 2: 302.096
-   Sample 3: 297.518
-   Sample 4: 301.542
-   Sample 5: 300.728

Prediction statistics:
Minimum: 295.58
Maximum: 304.44
Mean: 299.98
Standard deviation: 1.78

Comparison with training data:
Original range: 295.30 - 304.50
Original mean: 300.01

✅ Predictions were generated successfully!
Using ensemble model with approximate accuracy: 0.9256
